In [1]:
import sqlite3
import pandas as pd
import os
from datetime import datetime

In [2]:
LEGACY_DB   = "data/legacy.db"
MIGRATED_DB = "data/migrated.db"
EXPORT_DIR  = "data/exports"
REPORT_PATH = "data/migration_reconciliation_report.txt"
os.makedirs(EXPORT_DIR, exist_ok=True)
 
src = sqlite3.connect(LEGACY_DB)
dst = sqlite3.connect(MIGRATED_DB)
 
report_lines = []

In [3]:
def log(msg: str = ""):
    print(msg)
    report_lines.append(msg)
 
def section(title: str):
    log()
    log("=" * 60)
    log(f"  {title}")
    log("=" * 60)

In [4]:
tables = ["customers", "accounts", "billing_cycles", "payments"]
count_results = []
 
for table in tables:
    src_count = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table}", src)["n"][0]
    dst_count = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table}", dst)["n"][0]
    dropped   = src_count - dst_count
    drop_pct  = round(dropped / src_count * 100, 2) if src_count > 0 else 0
    status    = "⚠ DISCREPANCY" if dropped > 0 else "✓ OK"
    count_results.append({
        "table":       table,
        "source_rows": src_count,
        "target_rows": dst_count,
        "dropped":     dropped,
        "drop_pct":    drop_pct,
        "status":      status
    })
    log(f"  {table:<20} Source: {src_count:>8,}  Target: {dst_count:>8,}  Dropped: {dropped:>6,} ({drop_pct}%)  {status}")
 
df_counts = pd.DataFrame(count_results)
df_counts.to_csv(f"{EXPORT_DIR}/recon_record_counts.csv", index=False)

  customers            Source:   50,000  Target:   48,998  Dropped:  1,002 (2.0%)  ⚠ DISCREPANCY
  accounts             Source:   50,000  Target:   49,009  Dropped:    991 (1.98%)  ⚠ DISCREPANCY
  billing_cycles       Source:  500,000  Target:  490,081  Dropped:  9,919 (1.98%)  ⚠ DISCREPANCY
  payments             Source:  449,773  Target:  440,608  Dropped:  9,165 (2.04%)  ⚠ DISCREPANCY


In [6]:
id_cols = {
    "customers":     "customer_id",
    "accounts":      "account_id",
    "billing_cycles":"cycle_id",
    "payments":      "payment_id"
}
 
for table, id_col in id_cols.items():
    src_ids = set(pd.read_sql(f"SELECT {id_col} FROM {table}", src)[id_col])
    dst_ids = set(pd.read_sql(f"SELECT {id_col} FROM {table}", dst)[id_col])
    missing = src_ids - dst_ids
    log(f"  {table:<20} Missing IDs: {len(missing):,}")
 
    if missing:
        df_missing = pd.DataFrame({id_col: list(missing)})
        df_missing["table"] = table
        df_missing.to_csv(f"{EXPORT_DIR}/missing_{table}.csv", index=False)
        log(f"    → Exported: missing_{table}.csv")

  customers            Missing IDs: 1,002
    → Exported: missing_customers.csv
  accounts             Missing IDs: 991
    → Exported: missing_accounts.csv
  billing_cycles       Missing IDs: 9,919
    → Exported: missing_billing_cycles.csv
  payments             Missing IDs: 9,165
    → Exported: missing_payments.csv


In [5]:
q_billing_recon = """
    SELECT
        s.cycle_id,
        s.account_id,
        s.amount_due    AS source_amount,
        d.amount_due    AS target_amount,
        ROUND(s.amount_due - d.amount_due, 2) AS discrepancy
    FROM billing_cycles s
    JOIN billing_cycles d ON s.cycle_id = d.cycle_id
    WHERE ABS(s.amount_due - d.amount_due) > 0.01
"""
 
# Read from both connections separately then merge
src_cycles = pd.read_sql("SELECT cycle_id, account_id, amount_due FROM billing_cycles", src)
dst_cycles = pd.read_sql("SELECT cycle_id, amount_due FROM billing_cycles", dst)
 
merged_cycles = src_cycles.merge(dst_cycles, on="cycle_id", suffixes=("_source", "_target"))
merged_cycles["discrepancy"] = (merged_cycles["amount_due_source"] - merged_cycles["amount_due_target"]).round(2)
discrepant_cycles = merged_cycles[merged_cycles["discrepancy"].abs() > 0.01].copy()
 
log(f"  Billing cycles with amount discrepancy: {len(discrepant_cycles):,}")
log(f"  Total $$$ discrepancy: €{discrepant_cycles['discrepancy'].sum():,.2f}")
log(f"  Max single discrepancy: €{discrepant_cycles['discrepancy'].abs().max():,.2f}")
 
discrepant_cycles.to_csv(f"{EXPORT_DIR}/recon_billing_discrepancies.csv", index=False)
log(f"  → Exported: recon_billing_discrepancies.csv")

  Billing cycles with amount discrepancy: 5,030
  Total $$$ discrepancy: €-643.91
  Max single discrepancy: €15.00
  → Exported: recon_billing_discrepancies.csv


In [7]:
src_pay = pd.read_sql("SELECT payment_id, account_id, amount_paid FROM payments", src)
dst_pay = pd.read_sql("SELECT payment_id, amount_paid FROM payments", dst)
 
merged_pay = src_pay.merge(dst_pay, on="payment_id", suffixes=("_source", "_target"))
merged_pay["discrepancy"] = (merged_pay["amount_paid_source"] - merged_pay["amount_paid_target"]).round(2)
discrepant_pay = merged_pay[merged_pay["discrepancy"].abs() > 0.01].copy()
 
log(f"  Payments with amount discrepancy: {len(discrepant_pay):,}")
log(f"  Total $$$ discrepancy: €{discrepant_pay['discrepancy'].sum():,.2f}")
 
discrepant_pay.to_csv(f"{EXPORT_DIR}/recon_payment_discrepancies.csv", index=False)
log(f"  → Exported: recon_payment_discrepancies.csv")
 

  Payments with amount discrepancy: 4,400
  Total $$$ discrepancy: €-1,117.07
  → Exported: recon_payment_discrepancies.csv


In [8]:
date_checks = [
    ("billing_cycles", "cycle_start"),
    ("billing_cycles", "cycle_end"),
    ("billing_cycles", "due_date"),
    ("payments",       "payment_date"),
    ("accounts",       "last_payment_date"),
    ("customers",      "account_open_date"),
]
 
date_results = []
for table, col in date_checks:
    q = f"""
        SELECT COUNT(*) AS bad_dates
        FROM {table}
        WHERE {col} IS NOT NULL
          AND {col} NOT LIKE '____-__-__'
    """
    bad = pd.read_sql(q, dst)["bad_dates"][0]
    status = "⚠ BAD DATES FOUND" if bad > 0 else "✓ OK"
    log(f"  {table}.{col:<20} Bad dates: {bad:>6,}  {status}")
    date_results.append({"table": table, "column": col, "bad_dates": bad})
 
pd.DataFrame(date_results).to_csv(f"{EXPORT_DIR}/recon_date_validation.csv", index=False)
log(f"  → Exported: recon_date_validation.csv")

  billing_cycles.cycle_start          Bad dates:      0  ✓ OK
  billing_cycles.cycle_end            Bad dates:      0  ✓ OK
  billing_cycles.due_date             Bad dates:      0  ✓ OK
  payments.payment_date         Bad dates:      0  ✓ OK
  accounts.last_payment_date    Bad dates:      0  ✓ OK
  customers.account_open_date    Bad dates:      0  ✓ OK
  → Exported: recon_date_validation.csv


In [9]:
null_checks = [
    ("customers",      "first_name"),
    ("customers",      "email"),
    ("accounts",       "current_balance"),
    ("billing_cycles", "amount_due"),
    ("payments",       "amount_paid"),
]
 
null_results = []
for table, col in null_checks:
    q = f"SELECT COUNT(*) AS nulls FROM {table} WHERE {col} IS NULL"
    n = pd.read_sql(q, dst)["nulls"][0]
    status = "⚠ NULLS FOUND" if n > 0 else "✓ OK"
    log(f"  {table}.{col:<25} Nulls: {n:>6,}  {status}")
    null_results.append({"table": table, "column": col, "null_count": n})
 
pd.DataFrame(null_results).to_csv(f"{EXPORT_DIR}/recon_null_injection.csv", index=False)
log(f"  → Exported: recon_null_injection.csv")

  customers.first_name                Nulls:    228  ⚠ NULLS FOUND
  customers.email                     Nulls:      0  ✓ OK
  accounts.current_balance           Nulls:      0  ✓ OK
  billing_cycles.amount_due                Nulls:      0  ✓ OK
  payments.amount_paid               Nulls:      0  ✓ OK
  → Exported: recon_null_injection.csv


In [10]:
src_bal = pd.read_sql("SELECT account_id, current_balance FROM accounts", src)
dst_bal = pd.read_sql("SELECT account_id, current_balance FROM accounts", dst)
 
merged_bal = src_bal.merge(dst_bal, on="account_id", suffixes=("_source", "_target"))
merged_bal["drift"] = (merged_bal["current_balance_source"] - merged_bal["current_balance_target"]).round(2)
drifted = merged_bal[merged_bal["drift"].abs() > 0.01]
 
log(f"  Accounts with balance drift: {len(drifted):,}")
log(f"  Total balance drift: €{drifted['drift'].sum():,.2f}")
log(f"  Max drift on single account: €{drifted['drift'].abs().max():,.2f}")
 
drifted.to_csv(f"{EXPORT_DIR}/recon_balance_drift.csv", index=False)
log(f"  → Exported: recon_balance_drift.csv")

  Accounts with balance drift: 510
  Total balance drift: €155.08
  Max drift on single account: €15.00
  → Exported: recon_balance_drift.csv


In [11]:
src_billing_totals = pd.read_sql("""
    SELECT account_id,
           COUNT(*)         AS src_cycle_count,
           SUM(amount_due)  AS src_total_billed
    FROM billing_cycles GROUP BY account_id
""", src)
 
dst_billing_totals = pd.read_sql("""
    SELECT account_id,
           COUNT(*)         AS dst_cycle_count,
           SUM(amount_due)  AS dst_total_billed
    FROM billing_cycles GROUP BY account_id
""", dst)
 
master = src_billing_totals.merge(dst_billing_totals, on="account_id", how="outer")
master["cycle_count_diff"]  = master["src_cycle_count"] - master["dst_cycle_count"]
master["billing_amount_diff"] = (master["src_total_billed"] - master["dst_total_billed"]).round(2)
master["has_discrepancy"] = (
    (master["cycle_count_diff"].fillna(0) != 0) |
    (master["billing_amount_diff"].fillna(0).abs() > 0.01)
).astype(int)
 
master.to_csv(f"{EXPORT_DIR}/recon_master.csv", index=False)
log(f"  Master reconciliation: {len(master):,} accounts")
log(f"  Accounts with any discrepancy: {master['has_discrepancy'].sum():,}")
log(f"  → Exported: recon_master.csv")

  Master reconciliation: 50,000 accounts
  Accounts with any discrepancy: 13,049
  → Exported: recon_master.csv


In [12]:
report_lines.insert(0, f"MIGRATION RECONCILIATION REPORT")
report_lines.insert(1, f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
report_lines.insert(2, "=" * 60)
 
with open(REPORT_PATH, "w") as f:
    f.write("\n".join(report_lines))
 
print(f"\n✅ Full report written to: {REPORT_PATH}")
src.close()
dst.close()


✅ Full report written to: data/migration_reconciliation_report.txt
